In [ ]:
# Centralized Task Logger Utility
# Imports and environment setup
import logging
import os
import sys

# Ensure UTF-8 output on Windows consoles
if sys.platform == "win32":
    sys.stdout.reconfigure(encoding="utf-8")

In [ ]:
def get_task_logger(notebook_name_override=None, log_dir_override=None):
    """
    Dynamically creates a unique log file for notebook or task execution.
    Streams logs simultaneously to:
      1. Persistent log file (DBFS /tmp/etl_task_logs or local logs/etl_task_logs)
      2. Console / Notebook cell output (sys.stdout)
    """
    notebook_name = notebook_name_override
    run_id = "interactive"

    # 1. Access Databricks internal context to get Notebook Name and Job Run ID
    if not notebook_name:
        try:
            from databricks.sdk.runtime import dbutils

            # Access metadata of the Databricks notebook
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

            path_opt = ctx.notebookPath()
            nb_path = path_opt.get() if hasattr(path_opt, "get") and path_opt.isDefined() else str(path_opt)
            notebook_name = nb_path.split("/")[-1] if nb_path else "unknown_notebook"

            # Extract unique Run ID (falls back to multitaskParentRunId or 'interactive')
            run_id_opt = ctx.currentRunId()
            if hasattr(run_id_opt, "isDefined") and run_id_opt.isDefined():
                run_id = str(run_id_opt.get())
            else:
                tags = ctx.tags()
                multitask_id = tags.get("multitaskParentRunId") if hasattr(tags, "get") else None
                if hasattr(multitask_id, "isDefined") and multitask_id.isDefined():
                    run_id = str(multitask_id.get())
                else:
                    run_id = "interactive"
        except Exception:
            # Safety fallback if context extraction fails or running locally
            notebook_name = "etl_task"
            run_id = "standalone"

    # 2. Define the central logging directory with permission fallbacks
    if log_dir_override:
        base_log_dir = log_dir_override
    elif os.path.exists("/dbfs"):
        base_log_dir = "/dbfs/tmp/etl_task_logs"
    else:
        base_log_dir = os.path.join(os.getcwd(), "logs", "etl_task_logs")

    try:
        os.makedirs(base_log_dir, exist_ok=True)
    except Exception:
        base_log_dir = os.path.join(os.getcwd(), "logs", "etl_task_logs")
        os.makedirs(base_log_dir, exist_ok=True)

    # Define the absolute distinct path for THIS specific run
    log_filename = f"{notebook_name}_{run_id}.log"
    full_log_path = os.path.join(base_log_dir, log_filename)

    # 3. Initialize the logger named after the notebook
    logger = logging.getLogger(notebook_name)

    # Avoid duplicate handlers on re-runs in interactive notebook cells
    if not logger.handlers:
        logger.setLevel(logging.INFO)

        formatter = logging.Formatter(
            fmt="%(asctime)s | %(levelname)-8s | %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        )

        # File Handler (persistent disk log)
        file_handler = logging.FileHandler(full_log_path, mode="a", encoding="utf-8")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

        # Console Handler (visible in terminal and Databricks cell output)
        console_handler = logging.StreamHandler(sys.stdout)
        console_handler.setFormatter(formatter)
        logger.addHandler(console_handler)

        logger.propagate = False

    return logger

In [ ]:
# Interactive Self-Test (uncomment to test directly)
# test_logger = get_task_logger()
# test_logger.info("Self-test: Logger initialized successfully!")